In [2]:
library(tidyverse)
library(dbplyr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




In [3]:
# Read in data
data <- read.csv('data/destinations_wide_detail.csv', header = TRUE)

In [4]:
head(data)

,person_id,NCCIS_ACADYR,Intended_destination,X9,X10,X11,X12,X1,X2,X3,X4,X5,X6,X7,X8
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
2,00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017/2018,NA,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
3,0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018/2019,NA,Education,Education,Education,Education,Education,Education,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET
4,0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2018/2019,NA,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education
5,00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,2017/2018,NA,NA,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,NA
6,0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,2017/2018,NA,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education,Education


In [5]:
data |>
    filter(person_id == '0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D')

person_id,NCCIS_ACADYR,Intended_destination,X9,X10,X11,X12,X1,X2,X3,X4,X5,X6,X7,X8
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018/2019,NA,Education,Education,Education,Education,Education,Education,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET


## Prep data

In [3]:
data |>
    nrow()

[1] 17543

In [6]:
# drop all NAs 
destinations_seq <- data |>
    filter(!if_all(`X9`:`X8`, is.na))

In [7]:
destinations_seq |>
    nrow()

[1] 17361

## add activity counts per row

In [8]:
destinations_counts <- destinations_seq |>
    rowwise() |>
    mutate(
    Education = sum(c_across(`X9`:`X8`) == "Education", na.rm = TRUE),
    Employment = sum(c_across(`X9`:`X8`) == "Employment", na.rm = TRUE),
    Training = sum(c_across(`X9`:`X8`) == "Training", na.rm = TRUE),
    Refused = sum(c_across(`X9`:`X8`) == "Refused", na.rm = TRUE),
    # check starts with 
    NEET = sum(grepl("^NEET", c_across(`X9`:`X8`)), na.rm = TRUE),
    NA_count = sum(is.na(c_across(`X9`:`X8`)))
        ) |>
    ungroup()

## Classify records

In [24]:
destinations_counts |>
 filter(str_starts(X9, 'NEET') & (Education == 11 | Employment == 11 | Training == 11))

person_id,NCCIS_ACADYR,Intended_destination,X9,X10,X11,X12,X1,X2,X3,⋯,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count,Label
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<lgl>
00C46B3D88971DF1EBEFDEB5D91BD0E0DE6CAF37331A66161CDEDE3F6102ADC1,2018/2019,NA,NEET: Seeking EET,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,11,0,0,0,1,0,NA
0A942C4DE9995D01BEDFB2C69002ACBF364847CEAB5570A59AB858B67C6A2AB6,2018/2019,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,⋯,Training,Training,Training,0,0,11,0,1,0,NA
0CB91494A90E3006FE6A7D64C69ABC97753D9298C2AAD15EF1D50F4C0DBEAE6D,2018/2019,NA,NEET: Start date agreed,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,11,0,0,0,1,0,NA
0D01A21C41B86E59153B4CC7A68F41216823DF8A38E93F162DC51E6D13FD3FD0,2018/2019,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,⋯,Training,Training,Training,0,0,11,0,1,0,NA
0F566ADABABA91A707DE281C06841AB05A5E2BAA74C89008DEBE0642A5D75408,2018/2019,NA,NEET: Seeking EET,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,11,0,0,0,1,0,NA
13B42FD5D363A91665B4D5A6213D9E6C184089978CA688627CC14327A3050F36,2018/2019,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,⋯,Training,Training,Training,0,0,11,0,1,0,NA
13D68578D64C06294A3BBFEC424A116C599368433C106C68BEDDA12FB940F086,2018/2019,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,⋯,Training,Training,Training,0,0,11,0,1,0,NA
267F352151AA08472D066932A8F509A8FC276127755A29A9D1843E5151E9A341,2017/2018,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,⋯,Training,Training,Training,0,0,11,0,1,0,NA
2A08BEF4FE2B9FC31889FD839677330CDB1876071BAA631AD1AD8572B444BADF,2017/2018,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,⋯,Training,Training,Training,0,0,11,0,1,0,NA


In [6]:
destinations_counts |>
    filter(NEET == 1)

ERROR: Error: object 'destinations_counts' not found


In [25]:
destinations_counts_labelled <- destinations_counts |>
    mutate(Label = case_when(
        NEET == 0 & Refused == 0 & NA_count <= 4 ~ 'Steady EET',
        str_starts(X9, 'NEET') & (Education == 11 | Employment == 11 | Training == 11) ~ 'Steady EET',
        NEET == 0 & Refused == 0 & NA_count >= 5 ~ 'DROP',
        TRUE ~ 'Risky trajectory'
    )
          )

In [26]:
head(destinations_counts_labelled)

person_id,NCCIS_ACADYR,Intended_destination,X9,X10,X11,X12,X1,X2,X3,⋯,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count,Label
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018,NA,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,12,0,0,0,0,0,Steady EET
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017/2018,NA,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,12,0,0,0,0,0,Steady EET
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018/2019,NA,Education,Education,Education,Education,Education,Education,NEET: Not ready,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,6,0,0,0,6,0,Risky trajectory
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2018/2019,NA,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,12,0,0,0,0,0,Steady EET
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,2017/2018,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,NA,10,0,0,0,0,2,Steady EET
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,2017/2018,NA,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,12,0,0,0,0,0,Steady EET


In [27]:
destinations_counts_labelled |>
    filter(Label == 'Risky trajectory')

person_id,NCCIS_ACADYR,Intended_destination,X9,X10,X11,X12,X1,X2,X3,⋯,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count,Label
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018/2019,NA,Education,Education,Education,Education,Education,Education,NEET: Not ready,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,6,0,0,0,6,0,Risky trajectory
004C012DC8B27D82CC173C9590867A88C8B60E6F4C1EAE8AF73D53403DDD94B5,2018/2019,NA,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,⋯,NEET: Seeking EET,NEET: Seeking EET,NEET: Start date agreed,0,0,6,0,4,2,Risky trajectory
006E3883AF0261A8C303A137715C7DD62949D75FFD52F75536E0B79CECCC90E0,2017/2018,NA,Education,Education,Education,Education,Education,Education,NEET: Seeking EET,⋯,NEET: Other,NEET: Other,NEET: Start date agreed,6,0,0,0,6,0,Risky trajectory
00748B650176730AAF7158DCC2B2B974C8EB08D2BE8195BBD107F3B670619625,2017/2018,NA,NA,Employment,Employment,NEET: Start date agreed,Training,Training,Training,⋯,NEET: Start date agreed,NEET: Start date agreed,NEET: Start date agreed,0,2,4,0,5,1,Risky trajectory
00C0B218E62078028895BA8B9A96ED2A1B86CAF3C3B653E39FB4E3DAF90F32CF,2017/2018,NA,NA,Education,NEET: Start date agreed,Training,Training,Training,Training,⋯,Training,Training,NEET: Seeking EET,1,0,8,0,2,1,Risky trajectory
00D2D54BE54AA2D95A4868BF01E6147645BD35B1DE9161DC1AC32D1B2570E2E7,2017/2018,NA,NEET: Seeking EET,Employment,Employment,Employment,NEET: Seeking EET,NEET: Seeking EET,NEET: Pregnancy,⋯,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,0,3,0,0,9,0,Risky trajectory
00F9642EF1D0BB625578567B677490A2EEBE306021240F417B77AD63393D181B,2017/2018,NA,Education,Education,Education,Education,Education,Education,Education,⋯,Education,Education,NEET: Seeking EET,11,0,0,0,1,0,Risky trajectory
013DEF439FBB5DB5FAB8CA66A79222C46E828A7FBC647474C2F3FDCA63108D0D,2018/2019,NA,NA,Education,Education,Education,Education,Education,Education,⋯,Education,Education,NEET: Seeking EET,10,0,0,0,1,1,Risky trajectory
017AA9CBE090DF46D824482BB63267EDF494C0D84BE1C2BC263D2A4300750CD6,2017/2018,NA,Employment,Employment,Employment,Employment,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,⋯,Training,Training,Training,0,4,3,0,5,0,Risky trajectory


In [28]:
destinations_counts_labelled |>
    filter(str_starts(X9, 'NEET'))

person_id,NCCIS_ACADYR,Intended_destination,X9,X10,X11,X12,X1,X2,X3,⋯,X6,X7,X8,Education,Employment,Training,Refused,NEET,NA_count,Label
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>
00C46B3D88971DF1EBEFDEB5D91BD0E0DE6CAF37331A66161CDEDE3F6102ADC1,2018/2019,NA,NEET: Seeking EET,Education,Education,Education,Education,Education,Education,⋯,Education,Education,Education,11,0,0,0,1,0,Steady EET
00D2D54BE54AA2D95A4868BF01E6147645BD35B1DE9161DC1AC32D1B2570E2E7,2017/2018,NA,NEET: Seeking EET,Employment,Employment,Employment,NEET: Seeking EET,NEET: Seeking EET,NEET: Pregnancy,⋯,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,0,3,0,0,9,0,Risky trajectory
03771DA89236BFA5902AD7B4EAEE11A07C4B8FE9B29A5D2335694EF70B876064,2018/2019,NA,NEET: Not ready,Training,Training,Training,Training,Training,NEET: Not ready,⋯,NEET: Not ready,NEET: Seeking EET,NEET: Seeking EET,0,0,5,0,7,0,Risky trajectory
063216C2332702E36554685047F070ADE599365E8193BF57918CA73D5DE2E106,2018/2019,NA,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,Education,⋯,Education,Education,Education,6,0,0,0,6,0,Risky trajectory
069DA23DCA0EFEE4631389D6B2BE4E2E6C40D3F0A10B172A6BD7D63E509CC67B,2018/2019,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Employment,⋯,Employment,Employment,Employment,0,6,5,0,1,0,Risky trajectory
0A2D7A763027FB15C7F1AF60925BEFD106F1D989E6994ACD2EC92CB9B3924E21,2018/2019,NA,NEET: Seeking EET,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,NEET: Teen parent,Training,Training,⋯,Training,Training,Training,0,0,7,0,5,0,Risky trajectory
0A942C4DE9995D01BEDFB2C69002ACBF364847CEAB5570A59AB858B67C6A2AB6,2018/2019,NA,NEET: Seeking EET,Training,Training,Training,Training,Training,Training,⋯,Training,Training,Training,0,0,11,0,1,0,Steady EET
0BA571FAB8736DC2F5593C7CF04A1E755E046E4A8872FB4FD224C7049107C7BE,2017/2018,NA,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,NEET: Seeking EET,⋯,Refused,Refused,Refused,0,0,0,3,9,0,Risky trajectory
0C824E388720F4BE7B318BE44F24E5B388D46DB77AB71198426BA62C12092F7A,2017/2018,NA,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Not ready,NEET: Illness,NEET: Illness,NEET: Illness,⋯,NEET: Illness,NEET: Illness,NEET: Illness,0,0,0,0,12,0,Risky trajectory


In [29]:
destinations_counts_labelled |>
    group_by(Label) |>
    tally()


Label,n
<chr>,<int>
DROP,294
Risky trajectory,1361
Steady EET,15706


In [30]:
write.csv(destinations_counts_labelled, "data/manual_classification.csv", row.names = FALSE)

## Extract person IDs

In [31]:
person_ids <- destinations_counts_labelled |>
    select(person_id, NCCIS_ACADYR)

In [32]:
head(person_ids)

person_id,NCCIS_ACADYR
<chr>,<chr>
0002F12C1F92EC01CB0C05D441AA6D7AE16E9CD131BB64885744D08BA74E1FDD,2017/2018
00034E776A98176C2B6075634BC4A614400F32AA9C4D599FDDE156541F1F9A29,2017/2018
0004E60FEFC16781A48973C80F782B233869629C9CE44B741E2D2043047FCB1D,2018/2019
0013020C6ED2FF78BCA8E5F2D0B9B16E66C50B6A2AABB754FB63314CFE3A2EC7,2018/2019
00149D26484ED3BE2DEF154B8DB53A36B1A9FF0839C447A2CE19B0750225BA66,2017/2018
0014DF7EADF7C22A0FD2F6862B6F06778B199F1F2D61242125DA8EC539E6275D,2017/2018


In [33]:
write.csv(person_ids, "data/person_ids.csv", row.names = FALSE)